# Evaluate DAN Model — Webcam Fine-tune Result
So sánh **Original** vs **Finetuned** trên RAF-DB test set.

In [ ]:
import os, sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root
root = os.getcwd()
if root not in sys.path:
    sys.path.insert(0, root)
%matplotlib inline

In [ ]:
EMOTIONS = ['Surprise', 'Fear', 'Disgust', 'Happiness', 'Sadness', 'Anger', 'Neutral']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class DAN(nn.Module):
    def __init__(self, num_class=7, num_head=4):
        super().__init__()
        resnet = models.resnet18(weights=None)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.num_head = num_head
        self.conv_att = nn.Conv2d(512, self.num_head, kernel_size=1)
        self.fc = nn.Linear(512, num_class)
        self.bn = nn.BatchNorm1d(num_class)
    def forward(self, x):
        x = self.features(x)
        att_map = self.conv_att(x)
        att_map = att_map.view(att_map.size(0), self.num_head, -1)
        att_map = F.softmax(att_map, dim=2)
        att_map = att_map.view(att_map.size(0), self.num_head, x.size(2), x.size(3))
        x_flat = x.view(x.size(0), 1, x.size(1), -1)
        att_flat = att_map.view(att_map.size(0), self.num_head, 1, -1)
        wf = (x_flat * att_flat).sum(dim=-1)
        ff = wf.mean(dim=1)
        return self.bn(self.fc(ff))

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_set = datasets.ImageFolder(root=os.path.join(root, 'data/DATASET/test'), transform=transform)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=0)
print(f'Test samples: {len(test_set)}')

In [ ]:
def evaluate(model, loader):
    model.eval()
    conf = np.zeros((7, 7), dtype=int)
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            for i in range(labels.size(0)):
                conf[labels[i], preds[i]] += 1
    accs = [conf[i,i]/conf[i].sum() if conf[i].sum() > 0 else 0 for i in range(7)]
    overall = conf.trace() / conf.sum()
    return conf, accs, overall

def plot_results(name, conf, accs, overall):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{name} — Overall: {overall*100:.2f}%', fontsize=14)
    
    # Bar chart
    colors = ['#4CAF50' if a >= overall else '#FF9800' for a in accs]
    ax1.bar(EMOTIONS, [a*100 for a in accs], color=colors)
    ax1.axhline(y=overall*100, color='red', linestyle='--', label=f'Overall {overall*100:.1f}%')
    ax1.set_ylabel('Accuracy (%)')
    ax1.set_ylim(0, 100)
    ax1.tick_params(axis='x', rotation=30)
    ax1.legend()
    for i, a in enumerate(accs):
        ax1.text(i, a*100 + 1, f'{a*100:.1f}%', ha='center', fontsize=9)
    
    # Confusion matrix
    sns.heatmap(conf, annot=True, fmt='d', cmap='Blues', xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax2)
    ax2.set_xlabel('Predicted')
    ax2.set_ylabel('True')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Evaluate ORIGINAL
orig_path = os.path.join(root, 'outputs/models/best_dan_model.pth')
model = DAN(num_class=7, num_head=4).to(device)
sd = torch.load(orig_path, map_location=device)
if any(k.startswith('module.') for k in sd):
    sd = {k.replace('module.', ''): v for k, v in sd.items()}
model.load_state_dict(sd)
conf_orig, accs_orig, overall_orig = evaluate(model, test_loader)
plot_results('ORIGINAL MODEL', conf_orig, accs_orig, overall_orig)

In [ ]:
# Evaluate FINETUNED
ft_path = os.path.join(root, 'outputs/models/best_dan_model_finetuned.pth')
if not os.path.exists(ft_path):
    print('Finetuned model not found!')
else:
    model = DAN(num_class=7, num_head=4).to(device)
    sd = torch.load(ft_path, map_location=device)
    if any(k.startswith('module.') for k in sd):
        sd = {k.replace('module.', ''): v for k, v in sd.items()}
    model.load_state_dict(sd)
    conf_ft, accs_ft, overall_ft = evaluate(model, test_loader)
    plot_results('FINETUNED MODEL', conf_ft, accs_ft, overall_ft)

In [ ]:
# Compare side-by-side
if os.path.exists(ft_path):
    print('='*60)
    print(f'  {"Class":12s} {"Original":>10s} {"Finetuned":>10s} {"Change":>10s}')
    print('='*60)
    for i in range(7):
        orig = accs_orig[i] * 100
        ft = accs_ft[i] * 100
        diff = ft - orig
        sign = '+' if diff >= 0 else ''
        print(f'  {EMOTIONS[i]:12s} {orig:>8.2f}%  {ft:>8.2f}%  {sign}{diff:>7.2f}%')
    print('='*60)
    diff = overall_ft - overall_orig
    print(f'  {"OVERALL":12s} {overall_orig*100:>8.2f}%  {overall_ft*100:>8.2f}%  {"+" if diff>=0 else ""}{diff*100:.2f}%')
    print('='*60)